# Crawler batching — smoke trước khi chạy toàn range
Sau git pull, restart runtime nếu crawler đã được import. Chọn đúng Drive root cũ. Cấu hình giữ paths tương đối; dữ liệu bền vững ở runtime/, scratch SSD chỉ là staging. RUN_FULL_RANGE mặc định False.


In [ ]:
import os
from pathlib import Path
from google.colab import drive

mount_point = input('Mount point Drive trong runtime Colab: ').strip()
drive.mount(mount_point)

# Nhập đường dẫn thư mục dự án trên Drive của bạn
project_dir = input('Thư mục dự án đã tồn tại trên Drive (dùng đúng root smoke cũ): ').strip()
project_root = Path(project_dir).expanduser()
assert project_dir and project_root.is_dir(), 'Chọn thư mục đã tồn tại trên Drive'

os.chdir(project_root)
print(f'Thư mục làm việc hiện tại: {Path.cwd()}')

In [ ]:
from pathlib import Path
import subprocess, sys

repo_dir = Path('code')
repo_url = 'https://github.com/NewB-S2deeC0de/arXiv.git'

if not repo_dir.exists():
  print('Đang clone repo mới từ GitHub...')
  subprocess.run(['git', 'clone', repo_url, str(repo_dir)], check=True)
else:
  print('Đang cập nhật repo từ GitHub...')
  subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only'], check=True)

print('Cài đặt dependencies...')
subprocess.run(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '-r',
        str(repo_dir / 'src/requirements.txt'),
    ],
    check=True,
)

# Thêm module crawler vào sys.path
src_path = str((repo_dir / 'src').resolve())
if src_path not in sys.path:
  sys.path.insert(0, src_path)

# Giữ mọi project path tương đối; symlink trỏ tới SSD của runtime hiện tại.
import tempfile
scratch_dir = Path('scratch')
if scratch_dir.is_symlink() and not scratch_dir.exists():
    scratch_dir.unlink()  # Chỉ bỏ link hết hiệu lực, không xóa dữ liệu đích.
if not scratch_dir.exists():
    scratch_dir.symlink_to(tempfile.mkdtemp(prefix='arxiv-crawler-'), target_is_directory=True)
assert scratch_dir.is_dir()
print('Scratch SSD đã sẵn sàng qua project path: scratch/')


In [ ]:
from crawler.config import load_config
from crawler.secrets import load_semantic_scholar_key

config_path = repo_dir / 'config/crawler.yaml'
config = load_config(config_path)

# Dùng đường dẫn tương đối qua symlink đã tạo trong cell trước
config.paths['scratch_root'] = 'scratch'
config.validate()

s2_key = load_semantic_scholar_key()
if s2_key:
  print('Semantic Scholar: Authentication = KEY (Đã nhận diện API key)')
else:
  raise RuntimeError('Thêm SEMANTIC_SCHOLAR_API_KEY vào Colab Secrets và bật notebook access trước khi chạy.')

rng = config.range
print(
    f"Phạm vi cào dữ liệu: {rng['start_month']}.{rng['start_id']:05d} đến"
    f" {rng['end_month']}.{rng['end_id']:05d}"
)
print('Discovery batch:', config.network['discovery_batch_size'], 'Version metadata batch:', config.network['metadata_batch_size'])


In [ ]:
from crawler.progress import ProgressStore
import pandas as pd

store = ProgressStore(config.paths['persistent_root'])
counts = store.status_counts()

# Hiển thị bảng tổng hợp tiến độ dạng DataFrame
df_status = pd.DataFrame(counts).fillna(0).astype(int)
print('=== TIẾN ĐỘ THU THẬP DỮ LIỆU HIỆN TẠI ===')
display(df_status)

In [ ]:
RUN_FULL_RANGE = False  # Chỉ bật sau khi smoke + interrupt/resume đạt.
LIMIT = None if RUN_FULL_RANGE else 10
from crawler.pipeline import run_pipeline

# Đổi thành True nếu phiên trước bị ngắt đột ngột và dính file lock
TAKEOVER_STALE = False

print('Phạm vi phiên chạy:', '5.000 ID' if RUN_FULL_RANGE else 'smoke 10 ID')
summary = await run_pipeline(
    config=config,
    stages={'arxiv', 's2'},
    resume=True,
    limit=LIMIT,
    takeover_stale=TAKEOVER_STALE,
)

print('\n=== HOÀN TẤT PHIÊN CHẠY ===')
for k, v in summary.items():
  if k != 'events':
    print(f'{k}: {v}')

In [ ]:
# Chỉ đổi thành True khi bạn muốn thử lại các bài bị lỗi mạng hoặc 5xx
RETRY_FAILED = False

if RETRY_FAILED:
  import uuid
  from crawler.locking import DatasetLock

  print('Đang reset các task bị failed để chuẩn bị retry...')
  with DatasetLock(
      config.paths['persistent_root'],
      'retry-' + uuid.uuid4().hex,
      takeover_stale=TAKEOVER_STALE,
  ):
    store = ProgressStore(config.paths['persistent_root'])
    store.reset(
        'failed', stages={'source', 'metadata', 'references', 'discovery'}
    )

  print('Đang chạy lại các task lỗi...')
  summary = await run_pipeline(
      config, {'arxiv', 's2'}, resume=True, limit=LIMIT, takeover_stale=TAKEOVER_STALE
  )
  print(summary)
else:
  print('Bỏ qua bước retry (RETRY_FAILED=False).')

In [ ]:
import uuid
from crawler.export import export_snapshot

run_id = 'full-5000-' + uuid.uuid4().hex[:8]
print(f'Đang xuất snapshot dataset với Run ID: {run_id}...')
manifest = export_snapshot(config, run_id)

print('\n=== XUẤT DATASET THÀNH CÔNG ===')
print(f"Tổng số bài báo (papers.parquet): {manifest['papers']}")
print(f"Tổng số liên kết trích dẫn (references.parquet): {manifest['references']}")
print(f"Đường dẫn thư mục xuất: {config.paths['persistent_root']}/exports/{run_id}")

In [ ]:
from pathlib import Path
import pyarrow.parquet as pq

export_dir = Path(config.paths['persistent_root']) / 'exports' / run_id

df_papers = pq.read_table(export_dir / 'papers.parquet').to_pandas()
df_refs = pq.read_table(export_dir / 'references.parquet').to_pandas()

print(f'=== DỮ LIỆU BẢNG PAPERS ({len(df_papers)} dòng) ===')
display(df_papers.head(5))

print(f'\n=== DỮ LIỆU BẢNG REFERENCES ({len(df_refs)} dòng) ===')
display(df_refs.head(5))

### Checkpoint và hiệu năng
Progress gồm jobs.csv và jobs.journal.jsonl; không xóa journal hoặc dùng code cũ sau nâng cấp. Reconcile đầy đủ chạy khi bắt đầu/resume; trong phiên chỉ xác minh result vừa publish. Batch metadata theo IDvN giữ ngày từng phiên bản. Disk metrics dùng baseline + delta. Sau cập nhật, restart kernel, dùng đúng root cũ và smoke/interrupt trước full range.
